In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import pandas as pd

## IMPORTANTE

Para no ejecutar el código mil veces, he ido guardando los archivos en csv, estos pesan mas de 100mb por tanto no los puedo subir al github. Es decir que como mínimo se deberá ejecutar el código 1 vez para generar estos archivos.

Si ya se han generado, hay que saltar las ejecuciones donde se generen, estas son las que llamen a la función "factorizacion_ponderada_SGD_con_mascara"

In [2]:
df = pd.read_csv("BBDD_100K/ratings.csv")

indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
matriz_usuario_pelicula=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')

def normalizar_datos(matriz_escasez):
    # Creamos una copia de la matriz para evitar modificar el original
    matriz_escasez_copy = matriz_escasez.copy()
    
    # Inicializamos StandardScaler sin centrado en 0 debido a NaNs
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Aplicamos la normalización solo en las columnas que tienen datos no NaN
    for user_id in matriz_escasez_copy.index:
        # Seleccionamos las calificaciones del usuario (excluyendo NaNs)
        user_ratings = matriz_escasez_copy.loc[user_id].dropna()
        if not user_ratings.empty:
            # Normalizamos las calificaciones de este usuario
            normalized_ratings = scaler.fit_transform(user_ratings.values.reshape(-1, 1)).flatten()
            # Colocamos los valores normalizados en la matriz original, manteniendo NaNs donde no hay calificaciones
            matriz_escasez_copy.loc[user_id, user_ratings.index] = normalized_ratings
    
    # Llenamos los NaNs con 0
    matriz_escasez_copy = matriz_escasez_copy.fillna(0)
    return matriz_escasez_copy

matriz_normalizada = normalizar_datos(matriz_usuario_pelicula)
matriz_normalizada.head(20)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,0.000000,-0.458937,0.000000,0.000000,-0.458937,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.371391,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,0.0,-0.581226,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.958138,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.442374,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-1.636784,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Factorización ponderada de matrices

In [3]:
# Función para inicializar los factores de usuario y película
def inicializar_factores(num_usuarios, num_items, num_factors):
    # Inicializa las matrices U y V con valores aleatorios pequeños
    U = np.random.normal(scale=0.01, size=(num_usuarios, num_factors))
    V = np.random.normal(scale=0.01, size=(num_items, num_factors))
    return U, V

# Función para aplicar WMF
def factorizacion_ponderada_SGD_con_mascara(matriz, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv):
    num_usuarios, num_items = matriz.shape
    U, V = inicializar_factores(num_usuarios, num_items, num_factors)

    # Crear una máscara donde las entradas existentes tienen peso 1, las faltantes 0
    mascara = (matriz != 0).astype(float)

    for iteracion in range(num_iteraciones):
        for i in range(num_usuarios):
            for j in range(num_items):
                # Actualizar solo las entradas observadas
                if mascara[i, j] == 1:
                    error = matriz[i, j] - np.dot(U[i, :], V[j, :])
                    U[i, :] += learning_rate * (error * V[j, :] - regularizacion * U[i, :])
                    V[j, :] += learning_rate * (error * U[i, :] - regularizacion * V[j, :])

        # Calcular el error cuadrático medio solo para las entradas observadas
        mse = np.mean((mascara * (matriz - (U @ V.T))) ** 2)
        print(f"Iteración {iteracion + 1}/{num_iteraciones}, MSE: {mse:.4f}")

    # Generar las predicciones completas
    predicciones_completas = np.dot(U, V.T)

    # Convertir las predicciones a un DataFrame con índices y columnas originales
    predicciones_df = pd.DataFrame(predicciones_completas, index=matriz_normalizada.index, columns=matriz_normalizada.columns)

    # Guardar el DataFrame como un archivo CSV
    predicciones_df.to_csv(output_csv, index=True)
    print(f"Predicciones guardadas en {output_csv}")

    return U, V, predicciones_df

Vamos a predecir los valores faltantes

In [124]:
# Parámetros
output_csv = "FPM_100K/100K_usuario_pelicula_datos_simulados.csv"
num_factors = 10          # Número de factores latentes
num_iteraciones = 50      # Número de iteraciones
learning_rate = 0.05      # Tasa de aprendizaje
regularizacion = 0.1      # Parámetro de regularización

# Convertimos la matriz normalizada a numpy array
matriz_numpy = matriz_normalizada.values

# Aplicamos la factorización ponderada
U, V, predicciones_simuladas_df = factorizacion_ponderada_SGD_con_mascara(
    matriz_numpy, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv
)

# Predicciones completas
predicciones_completas = np.dot(U, V.T)

# Crear una máscara para identificar las entradas faltantes
mascara = (matriz_numpy != 0).astype(float)

# Predicciones solo para las entradas faltantes
predicciones_simuladas = (1 - mascara) * predicciones_completas

# Convertimos a DataFrame para visualizar mejor
predicciones_simuladas_df = pd.DataFrame(
    predicciones_simuladas, index=matriz_normalizada.index, columns=matriz_normalizada.columns
)

predicciones_simuladas_df.head()

Iteración 1/50, MSE: 0.0170
Iteración 2/50, MSE: 0.0168
Iteración 3/50, MSE: 0.0153
Iteración 4/50, MSE: 0.0142
Iteración 5/50, MSE: 0.0137
Iteración 6/50, MSE: 0.0133
Iteración 7/50, MSE: 0.0130
Iteración 8/50, MSE: 0.0126
Iteración 9/50, MSE: 0.0122
Iteración 10/50, MSE: 0.0118
Iteración 11/50, MSE: 0.0114
Iteración 12/50, MSE: 0.0110
Iteración 13/50, MSE: 0.0106
Iteración 14/50, MSE: 0.0103
Iteración 15/50, MSE: 0.0099
Iteración 16/50, MSE: 0.0096
Iteración 17/50, MSE: 0.0093
Iteración 18/50, MSE: 0.0091
Iteración 19/50, MSE: 0.0089
Iteración 20/50, MSE: 0.0087
Iteración 21/50, MSE: 0.0086
Iteración 22/50, MSE: 0.0085
Iteración 23/50, MSE: 0.0084
Iteración 24/50, MSE: 0.0083
Iteración 25/50, MSE: 0.0082
Iteración 26/50, MSE: 0.0081
Iteración 27/50, MSE: 0.0081
Iteración 28/50, MSE: 0.0080
Iteración 29/50, MSE: 0.0080
Iteración 30/50, MSE: 0.0080
Iteración 31/50, MSE: 0.0079
Iteración 32/50, MSE: 0.0079
Iteración 33/50, MSE: 0.0079
Iteración 34/50, MSE: 0.0078
Iteración 35/50, MSE: 0

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,-0.455120,-0.000000,-0.721271,0.040664,0.000000,-1.181791,-0.725035,-0.313448,0.024930,...,-0.050670,-0.167858,0.061086,0.074379,-0.053888,0.080850,-0.049847,-0.046171,-0.051143,0.083949
2,-0.143362,-0.276165,0.490238,-0.679712,0.264125,-0.304524,-0.301213,0.235791,0.147704,0.001992,...,-0.000376,-0.094343,0.042719,0.048999,-0.023295,0.030485,-0.023649,-0.037232,-0.033285,-0.032309
3,0.033611,0.021319,-0.262676,0.838324,-0.580176,0.039841,0.389541,0.565227,0.354291,0.392834,...,0.005350,-0.039019,0.034336,0.026906,-0.010671,0.012200,-0.009983,-0.021346,-0.019255,0.034431
4,0.012380,0.302443,-0.060284,-0.556698,-0.091188,-0.043897,0.014338,0.277513,-0.255645,-0.103885,...,-0.011091,-0.000565,-0.012823,-0.001428,-0.015175,0.006733,0.004649,-0.002874,-0.000253,-0.036454
5,-0.000000,-0.009127,0.395731,-0.772151,0.142612,0.196609,0.000589,0.232195,-0.753898,-0.076567,...,-0.061427,-0.168171,0.055554,0.060115,-0.057309,0.079139,-0.048598,-0.054913,-0.043042,0.039582


Juntamos ambas matrices

In [126]:
matriz_simulada = pd.read_csv("FPM_100K/100K_usuario_pelicula_datos_simulados.csv", index_col = 0)
# Aseguramos que los índices y columnas coincidan
matriz_simulada.columns = matriz_normalizada.columns
matriz_simulada.index = matriz_normalizada.index

matriz_completa = matriz_normalizada.copy()
matriz_completa = matriz_completa.where(matriz_completa != 0, matriz_simulada)

matriz_completa.head(6)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,-0.455120,-0.458937,-0.721271,0.040664,-0.458937,-1.181791,-0.725035,-0.313448,0.024930,...,-0.050670,-0.167858,0.061086,0.074379,-0.053888,0.080850,-0.049847,-0.046171,-0.051143,0.083949
2,-0.143362,-0.276165,0.490238,-0.679712,0.264125,-0.304524,-0.301213,0.235791,0.147704,0.001992,...,-0.000376,-0.094343,0.042719,0.048999,-0.023295,0.030485,-0.023649,-0.037232,-0.033285,-0.032309
3,0.033611,0.021319,-0.262676,0.838324,-0.580176,0.039841,0.389541,0.565227,0.354291,0.392834,...,0.005350,-0.039019,0.034336,0.026906,-0.010671,0.012200,-0.009983,-0.021346,-0.019255,0.034431
4,0.012380,0.302443,-0.060284,-0.556698,-0.091188,-0.043897,0.014338,0.277513,-0.255645,-0.103885,...,-0.011091,-0.000565,-0.012823,-0.001428,-0.015175,0.006733,0.004649,-0.002874,-0.000253,-0.036454
5,0.371391,-0.009127,0.395731,-0.772151,0.142612,0.196609,0.000589,0.232195,-0.753898,-0.076567,...,-0.061427,-0.168171,0.055554,0.060115,-0.057309,0.079139,-0.048598,-0.054913,-0.043042,0.039582
6,0.541621,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,-0.421036,-0.581226,...,-0.038638,-0.050267,0.006684,0.018785,-0.007868,0.022817,-0.010195,-0.000950,-0.004023,-0.005484


### Vamos a reescalar los valores, tanto los simulados como los normalizados sobre los valores originales

Primero los normalizados

In [127]:
def reescalar_matriz_normalizada(predicciones_normalizadas, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa sólo donde había datos originales
    predicciones_reescaladas = predicciones_normalizadas.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    predicciones_reescaladas = predicciones_reescaladas.where(~matriz_usuario_pelicula.isna(), np.nan)
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas

In [128]:
matriz_normalizada_reescalada = reescalar_matriz_normalizada(matriz_normalizada, matriz_usuario_pelicula)
matriz_normalizada_reescalada.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Luego los simulados

In [129]:
def reescalar_matriz_simulada(matriz_simulada, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Calculamos la media y desviación estándar de cada usuario ignorando los NaNs originales
    medias_usuarios = matriz_usuario_pelicula.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula.std(axis=1, skipna=True)
    
    # Aplicamos la transformación inversa a toda la matriz simulada
    predicciones_reescaladas = matriz_simulada.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    
    # Redondeamos y limitamos los valores dentro del rango permitido
    predicciones_reescaladas = predicciones_reescaladas.round()
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating)
    
    return predicciones_reescaladas


In [130]:
def reescalar_solo_simulados(matriz_simulada, matriz_usuario_pelicula_original, min_rating=0.5, max_rating=5.0):
    """
    Reescala solo los valores simulados en la matriz simulada, utilizando las estadísticas de la matriz original.
    """
    # Crear una máscara de los valores simulados (donde matriz_usuario_pelicula_original tiene NaN)
    mascara_simulados = matriz_usuario_pelicula_original.isna()

    # Verificar alineación
    if not matriz_simulada.index.equals(matriz_usuario_pelicula_original.index) or not matriz_simulada.columns.equals(matriz_usuario_pelicula_original.columns):
        raise ValueError("Índices o columnas de las matrices no están alineados.")

    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)

    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)

    # Crear una copia para trabajar únicamente con los valores simulados
    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulados] = np.nan  # Mantener solo los valores simulados

    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)

    return valores_simulados


In [131]:
# Reescalar solo los valores simulados
valores_simulados_reescalados = reescalar_solo_simulados(
    matriz_simulada=matriz_simulada,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    min_rating=0.5,
    max_rating=5.0
)

# Mostrar las primeras filas de la matriz reescalada con solo valores simulados
print("Valores Simulados Reescalados (solo simulados, 2 decimales):")
valores_simulados_reescalados.head(6)


Valores Simulados Reescalados (solo simulados, 2 decimales):


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,4.00,NaN,3.79,4.40,NaN,3.42,3.79,4.12,4.39,...,4.33,4.23,4.42,4.43,4.32,4.43,4.33,4.33,4.33,4.43
2,3.83,3.73,4.34,3.40,4.16,3.70,3.71,4.14,4.07,3.95,...,3.95,3.87,3.98,3.99,3.93,3.97,3.93,3.92,3.92,3.92
3,2.51,2.48,1.89,4.19,1.22,2.52,3.25,3.62,3.18,3.26,...,2.45,2.35,2.51,2.49,2.41,2.46,2.42,2.39,2.40,2.51
4,3.57,3.95,3.48,2.82,3.44,3.50,3.57,3.92,3.22,3.42,...,3.54,3.55,3.54,3.55,3.54,3.56,3.56,3.55,3.56,3.51
5,NaN,3.63,4.03,2.87,3.78,3.83,3.64,3.87,2.89,3.56,...,3.58,3.47,3.69,3.70,3.58,3.71,3.59,3.58,3.59,3.68
6,3.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.14,NaN,...,3.46,3.45,3.50,3.51,3.49,3.51,3.48,3.49,3.49,3.49


Matriz completa

In [132]:
# Aseguramos que las matrices tienen índices y columnas alineados
valores_simulados_reescalados.columns = matriz_normalizada.columns
valores_simulados_reescalados.index = matriz_normalizada.index

# Crear la matriz completa reescalada
matriz_completa_reescalada = matriz_normalizada_reescalada.copy()
matriz_completa_reescalada = matriz_completa_reescalada.where(~pd.isna(matriz_completa_reescalada), valores_simulados_reescalados)

matriz_completa_reescalada.head(6)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.00,4.00,4.00,3.79,4.40,4.00,3.42,3.79,4.12,4.39,...,4.33,4.23,4.42,4.43,4.32,4.43,4.33,4.33,4.33,4.43
2,3.83,3.73,4.34,3.40,4.16,3.70,3.71,4.14,4.07,3.95,...,3.95,3.87,3.98,3.99,3.93,3.97,3.93,3.92,3.92,3.92
3,2.51,2.48,1.89,4.19,1.22,2.52,3.25,3.62,3.18,3.26,...,2.45,2.35,2.51,2.49,2.41,2.46,2.42,2.39,2.40,2.51
4,3.57,3.95,3.48,2.82,3.44,3.50,3.57,3.92,3.22,3.42,...,3.54,3.55,3.54,3.55,3.54,3.56,3.56,3.55,3.56,3.51
5,4.00,3.63,4.03,2.87,3.78,3.83,3.64,3.87,2.89,3.56,...,3.58,3.47,3.69,3.70,3.58,3.71,3.59,3.58,3.59,3.68
6,3.95,4.00,5.00,3.00,5.00,4.00,4.00,3.00,3.14,3.00,...,3.46,3.45,3.50,3.51,3.49,3.51,3.48,3.49,3.49,3.49


Como en el ejemplo anterior con KNN voy a ver si la media de simulacion de los datos simulados para el usuario 442 que tenía originalmente una media de 1.275 se acercan o no

In [133]:
valores_simulados_usuario_442 = valores_simulados_reescalados.loc[442]
puntuaciones_simuladas = valores_simulados_usuario_442.dropna()
media_simulada = puntuaciones_simuladas.mean()

print(f"Media de puntuación simulada del usuario 442: {media_simulada:.2f}")


Media de puntuación simulada del usuario 442: 1.26


Los datos son muy similares, ahora tocaria mirar el accuracy prediciendo algunos valores originales aleatorios. En concreto voy a seleccionar un 10% aleatorio sobre la máscara de la matriz original para simular datos, ese 10% serán valores originales que voy a simular para posteriormente comparar, si los resultados son satisfactorios, la factorización ponderada de matrices desarrollada será válida

## Accuracy

Primero seleccionamos un 10% aleatorio de datos originales

In [134]:
def generar_factorizacion(matriz_original, mascara_simulacion, num_factors, num_iteraciones, learning_rate, regularizacion, output_csv="FPM_100K/evaluacion_10%.csv"):
    """
    Simula exclusivamente los valores seleccionados por la máscara.
    """
    # Crear una copia de la matriz y aplicar la máscara de simulación
    matriz_modificada = matriz_original.copy()
    matriz_modificada[mascara_simulacion] = 0  # Eliminar temporalmente los valores seleccionados para simulación
    matriz_modificada = matriz_modificada.fillna(0)  # Reemplazar NaN por 0 para evitar errores

    # Aplicar la factorización ponderada
    U, V,fpm = factorizacion_ponderada_SGD_con_mascara(matriz_modificada.values, num_factors, num_iteraciones, learning_rate, regularizacion,output_csv)

    predicciones_completas = np.dot(U, V.T)
    predicciones_completas_df = pd.DataFrame(predicciones_completas, index=matriz_original.index, columns=matriz_original.columns)
    
    return predicciones_completas_df

In [135]:
def recortar_por_mascara(predicciones_completas, mascara_simulacion):
    """
    Recorta las predicciones completas utilizando la máscara de simulación.
    Devuelve un DataFrame con NaN en las posiciones fuera de la máscara.

    Args:
    - predicciones_completas (DataFrame): Predicciones generadas para toda la matriz.
    - mascara_simulacion (DataFrame): Máscara booleana que indica las posiciones a conservar.

    Returns:
    - DataFrame: Predicciones recortadas según la máscara.
    """
    # Crear un DataFrame con NaN en todas las posiciones
    predicciones_recortadas = pd.DataFrame(
        np.nan, index=predicciones_completas.index, columns=predicciones_completas.columns
    )
    # Conservar solo las posiciones seleccionadas por la máscara
    predicciones_recortadas[mascara_simulacion] = predicciones_completas[mascara_simulacion]

    return predicciones_recortadas

In [136]:
def reescalar_simulados_con_mascara(matriz_simulada, matriz_usuario_pelicula_original, mascara_simulacion, min_rating=0.5, max_rating=5.0):
    """
    Reescala únicamente los valores simulados seleccionados por la máscara, utilizando las estadísticas de la matriz original.
    """
    # Verificar alineación
    if not matriz_simulada.index.equals(matriz_usuario_pelicula_original.index) or \
       not matriz_simulada.columns.equals(matriz_usuario_pelicula_original.columns):
        raise ValueError("Índices o columnas de las matrices no están alineados.")
    if not matriz_simulada.index.equals(mascara_simulacion.index) or \
       not matriz_simulada.columns.equals(mascara_simulacion.columns):
        raise ValueError("Índices o columnas de la máscara no están alineados con la matriz simulada.")

    # Calculamos la media y desviación estándar de cada usuario desde la matriz original
    medias_usuarios = matriz_usuario_pelicula_original.mean(axis=1, skipna=True)
    desviaciones_usuarios = matriz_usuario_pelicula_original.std(axis=1, skipna=True)

    # Manejar desviaciones estándar NaN o cero reemplazándolas con 1
    desviaciones_usuarios = desviaciones_usuarios.replace(0, 1).fillna(1)

    # Crear una copia para trabajar únicamente con los valores simulados según la máscara
    valores_simulados = matriz_simulada.copy()
    valores_simulados[~mascara_simulacion] = np.nan  # Mantener solo los valores seleccionados por la máscara

    # Aplicar el reescalado a los valores seleccionados
    valores_simulados = valores_simulados.mul(desviaciones_usuarios, axis=0).add(medias_usuarios, axis=0)
    valores_simulados = valores_simulados.clip(lower=min_rating, upper=max_rating)
    valores_simulados = valores_simulados.round(2)

    return valores_simulados


In [137]:
num_factors = 10
num_iteraciones = 50
learning_rate = 0.05
regularizacion = 0.1

mascara_original = ~matriz_usuario_pelicula.isna()
num_datos = mascara_original.sum().sum()
num_datos_a_simular = int(0.1 * num_datos)

indices_aleatorios = np.random.choice(
    mascara_original.stack()[mascara_original.stack()].index,
    size=num_datos_a_simular,
    replace=False
)

mascara_simulacion = pd.DataFrame(False, index=matriz_usuario_pelicula.index, columns=matriz_usuario_pelicula.columns)
for fila, columna in indices_aleatorios:
    mascara_simulacion.loc[fila, columna] = True

Vamos a mirar si los valores pertenecen a la máscara

In [138]:
num_valores_validos = mascara_original.sum().sum()
num_simulados = mascara_simulacion.sum().sum()
porcentaje_simulados = (num_simulados / num_valores_validos) * 100

print(f"Total de valores válidos en la matriz original: {num_valores_validos}")
print(f"Total de valores seleccionados para simulación: {num_simulados}")
print(f"Porcentaje de valores simulados: {porcentaje_simulados:.2f}%")

Total de valores válidos en la matriz original: 100836
Total de valores seleccionados para simulación: 10083
Porcentaje de valores simulados: 10.00%


In [139]:
predicciones_completas = generar_factorizacion(
    matriz_original=matriz_normalizada,
    mascara_simulacion=mascara_simulacion,
    num_factors=num_factors,
    num_iteraciones=num_iteraciones,
    learning_rate=learning_rate,
    regularizacion=regularizacion
)

Iteración 1/50, MSE: 0.0153
Iteración 2/50, MSE: 0.0152
Iteración 3/50, MSE: 0.0144
Iteración 4/50, MSE: 0.0131
Iteración 5/50, MSE: 0.0125
Iteración 6/50, MSE: 0.0121
Iteración 7/50, MSE: 0.0118
Iteración 8/50, MSE: 0.0115
Iteración 9/50, MSE: 0.0112
Iteración 10/50, MSE: 0.0108
Iteración 11/50, MSE: 0.0104
Iteración 12/50, MSE: 0.0100
Iteración 13/50, MSE: 0.0096
Iteración 14/50, MSE: 0.0092
Iteración 15/50, MSE: 0.0089
Iteración 16/50, MSE: 0.0085
Iteración 17/50, MSE: 0.0082
Iteración 18/50, MSE: 0.0080
Iteración 19/50, MSE: 0.0078
Iteración 20/50, MSE: 0.0076
Iteración 21/50, MSE: 0.0074
Iteración 22/50, MSE: 0.0073
Iteración 23/50, MSE: 0.0072
Iteración 24/50, MSE: 0.0071
Iteración 25/50, MSE: 0.0070
Iteración 26/50, MSE: 0.0070
Iteración 27/50, MSE: 0.0069
Iteración 28/50, MSE: 0.0069
Iteración 29/50, MSE: 0.0068
Iteración 30/50, MSE: 0.0068
Iteración 31/50, MSE: 0.0067
Iteración 32/50, MSE: 0.0067
Iteración 33/50, MSE: 0.0067
Iteración 34/50, MSE: 0.0067
Iteración 35/50, MSE: 0

In [140]:
predicciones_10_por_ciento = pd.read_csv("FPM_100K/evaluacion_10%.csv",index_col=0)
predicciones_10_por_ciento.columns = predicciones_10_por_ciento.columns.astype(int)

predicciones_recortadas = recortar_por_mascara(predicciones_10_por_ciento, mascara_simulacion)
predicciones_recortadas.head()

,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
valores_reescalados_con_mascara = reescalar_simulados_con_mascara(
    matriz_simulada=predicciones_recortadas,
    matriz_usuario_pelicula_original=matriz_usuario_pelicula,
    mascara_simulacion=mascara_simulacion
)

print("Valores reescalados con máscara:")
valores_reescalados_con_mascara.head()

Valores reescalados con máscara:


,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [142]:
def calcular_accuracy_solo_simulados(valores_originales, valores_simulados):
    # Máscara de valores simulados válidos (no NaN)
    mascara_simulados = ~valores_simulados.isna()

    # Extraer los valores simulados y sus correspondientes originales
    valores_simulados_filtrados = valores_simulados[mascara_simulados]
    valores_originales_filtrados = valores_originales[mascara_simulados]

    # Comparar cada valor simulado con el correspondiente original
    diferencias_relativas = np.abs((valores_simulados_filtrados - valores_originales_filtrados) / valores_originales_filtrados)

    accuracy_promedio = 100 - (diferencias_relativas.mean().mean() * 100)

    return accuracy_promedio

In [143]:
accuracy_promedio = calcular_accuracy_solo_simulados(matriz_usuario_pelicula, valores_reescalados_con_mascara)

print(f"Accuracy promedio basado en las posiciones simuladas: {accuracy_promedio:.2f}%")

Accuracy promedio basado en las posiciones simuladas: 66.98%
